# 02 Experiment：月总量预测模型 Benchmark

目标：基于 `train.csv / valid.csv / test.csv` 做可复现的模型实验。评估口径由 `CONFIG.forecast_offset` 指定：每月倒数第 N 个工作日预测当月总量，并用月总量 MAPE 做模型比较。

MLE 约束：训练、特征选择和调参只使用 train；valid 只用于模型选择；test 只用于最终报告。所有预测时不可见的结果列都从训练特征中排除。

## Block 1：数据接入

本 block 锁定数据契约、字段语义和泄露边界。输入表是月粒度 feature/target 表，每行代表一个 `bizym + forecast_offset` 锚点时的一次预测样本。

In [ ]:
from __future__ import annotations

# 中文说明：本 notebook 尽量保持自包含，便于后续复制实验和复查泄露边界。
from dataclasses import dataclass
from pathlib import Path
from typing import Any
import importlib.util
import itertools
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.font_manager as font_manager
import seaborn as sns

from sklearn.base import clone
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import (
    AdaBoostRegressor,
    BaggingRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.feature_selection import mutual_info_regression
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, HuberRegressor, Lasso, LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import ParameterGrid, TimeSeriesSplit
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor

try:
    from xgboost import XGBRegressor
except Exception:
    XGBRegressor = None

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", "{:.4f}".format)

def configure_chinese_font() -> None:
    """中文说明：自动选择本机可用中文字体，避免图例/标题中文显示为方块。"""
    preferred_fonts = [
        "Arial Unicode MS",
        "PingFang SC",
        "Heiti SC",
        "Songti SC",
        "Microsoft YaHei",
        "SimHei",
        "Noto Sans CJK SC",
        "WenQuanYi Zen Hei",
    ]
    available_fonts = {font.name for font in font_manager.fontManager.ttflist}
    for font_name in preferred_fonts:
        if font_name in available_fonts:
            plt.rcParams["font.sans-serif"] = [font_name, *plt.rcParams.get("font.sans-serif", [])]
            break
    plt.rcParams["axes.unicode_minus"] = False


sns.set_theme(style="whitegrid", palette="Set2")
configure_chinese_font()


In [ ]:
# 中文说明：集中管理实验配置，避免 notebook 隐式状态影响结果。
@dataclass(frozen=True)
class ExperimentConfig:
    data_dir: Path = Path("code/30d-jenny")
    target_col: str = "actual_month_total"
    forecast_offset: int = 10 # 倒数第几个工作日做全月预测
    primary_metric: str = "month_total_mape_pct"
    random_seed: int = 42
    cv_splits: int = 3
    n_iter: int = 20
    model_include: str = "all"
    use_pycaret_if_available: bool = True
    top_k_features: int = 8


CONFIG = ExperimentConfig()

TRAIN_PATH = CONFIG.data_dir / "train.csv"
VALID_PATH = CONFIG.data_dir / "valid.csv"
TEST_PATH = CONFIG.data_dir / "test.csv"

# 中文说明：这些列在预测时不可用，或属于标签/展示字段，严禁进入训练特征。
LEAKAGE_COLS = {
    "actual_month_total",
    "split",
}

# 中文说明：日期和标识列保留做展示，不直接作为数值特征；会拆出安全的日历特征。
IDENTIFIER_COLS = {"bizym", "forecast_offset", "anchor_date", "forecast_start_date"}

REQUIRED_COLUMNS = {
    "bizym",
    "forecast_offset",
    "actual_month_total",
    "workdays",
    "days",
    "prev_year_ym",
    "prev_2year_ym",
    "has_y1_y2_history",
    "anchor_date",
    "forecast_start_date",
    "anchor_wd_seq",
    "max_wd_seq",
    "remaining_workdays_after_anchor",
    "anchor_mtd_qty",
    "anchor_mtd_num_hosp",
    "split",
}

np.random.seed(CONFIG.random_seed)


In [ ]:
def resolve_data_path(path: Path) -> Path:
    """中文说明：同时支持从仓库根目录或 notebook 所在目录运行。"""
    if path.exists():
        return path
    local_path = Path(".") / path.name
    if local_path.exists():
        return local_path
    raise FileNotFoundError(f"找不到数据文件: {path}")


def load_split_csv(path: Path, expected_split: str) -> pd.DataFrame:
    """中文说明：读取并校验单个 split 的月粒度样本。"""
    resolved_path = resolve_data_path(path)
    df = pd.read_csv(resolved_path)
    missing_cols = sorted(REQUIRED_COLUMNS - set(df.columns))
    if missing_cols:
        raise ValueError(f"{resolved_path} 缺少必要字段: {missing_cols}")

    df = df.copy()
    df["anchor_date"] = pd.to_datetime(df["anchor_date"], errors="coerce")
    df["forecast_start_date"] = pd.to_datetime(df["forecast_start_date"], errors="coerce")
    if df[["anchor_date", "forecast_start_date"]].isna().any().any():
        raise ValueError(f"{resolved_path} 存在无法解析的日期")

    df["bizym"] = df["bizym"].astype(int)
    df["forecast_offset"] = df["forecast_offset"].astype(int)
    if not (df["split"] == expected_split).all():
        raise ValueError(f"{resolved_path} 的 split 标签不全是 {expected_split}")
    if (df[CONFIG.target_col] <= 0).any():
        raise ValueError(f"{resolved_path} 存在非正 target，无法计算 MAPE")
    return df.sort_values(["bizym", "forecast_offset"]).reset_index(drop=True)


def require_configured_forecast_offset(*dfs: pd.DataFrame) -> None:
    """中文说明：确保 02 指定的 offset 来自 01 导出的上游 offset list。"""
    available_offsets = sorted(set().union(*[set(df["forecast_offset"].astype(int)) for df in dfs]))
    if CONFIG.forecast_offset not in available_offsets:
        raise ValueError(
            f"CONFIG.forecast_offset={CONFIG.forecast_offset} 不在上游预测时间点列表 {available_offsets} 中；"
            f"请改成其中任意一个值。"
        )
    print(f"available forecast offsets from upstream: {available_offsets}; selected: {CONFIG.forecast_offset}")


def select_forecast_offset(df: pd.DataFrame) -> pd.DataFrame:
    out = df[df["forecast_offset"] == CONFIG.forecast_offset].copy()
    if out.empty:
        raise ValueError(f"split={df['split'].iloc[0] if not df.empty else 'unknown'} 没有 forecast_offset={CONFIG.forecast_offset} 的样本")
    return out.sort_values("bizym").reset_index(drop=True)


train_all_offsets = load_split_csv(TRAIN_PATH, "train")
valid_all_offsets = load_split_csv(VALID_PATH, "valid")
test_all_offsets = load_split_csv(TEST_PATH, "test")
require_configured_forecast_offset(train_all_offsets, valid_all_offsets, test_all_offsets)

train_df = select_forecast_offset(train_all_offsets)
valid_df = select_forecast_offset(valid_all_offsets)
test_df = select_forecast_offset(test_all_offsets)
all_df = pd.concat([train_df, valid_df, test_df], ignore_index=True).sort_values("bizym").reset_index(drop=True)

if not (train_df["bizym"].max() < valid_df["bizym"].min() < valid_df["bizym"].max() < test_df["bizym"].min()):
    raise ValueError("train/valid/test 月份顺序异常，可能存在时间泄露")

contract = pd.DataFrame({
    "字段": sorted(REQUIRED_COLUMNS),
    "用途": [
        "target/analysis-only" if col in LEAKAGE_COLS else "metadata" if col in IDENTIFIER_COLS else "candidate_feature"
        for col in sorted(REQUIRED_COLUMNS)
    ],
})

print(f"train: {train_df.shape}, {train_df['bizym'].min()} ~ {train_df['bizym'].max()}")
print(f"valid: {valid_df.shape}, {valid_df['bizym'].min()} ~ {valid_df['bizym'].max()}")
print(f"test : {test_df.shape}, {test_df['bizym'].min()} ~ {test_df['bizym'].max()}")
display(contract)


## Block 2：新建实验（特征选择 + 常见模型训练）

本 block 只使用指定预测 offset 锚点时已经可见的信息训练模型。`actual_month_total` 作为标签只用于训练目标和评估，不进入训练特征。

In [ ]:
def add_safe_features(df: pd.DataFrame) -> pd.DataFrame:
    """中文说明：只构造预测锚点时已经可见的安全特征。"""
    out = df.copy()
    out["year"] = out["bizym"] // 100
    out["month_num"] = out["bizym"] % 100
    out["anchor_day"] = out["anchor_date"].dt.day
    out["anchor_dow"] = out["anchor_date"].dt.dayofweek
    out["forecast_start_day"] = out["forecast_start_date"].dt.day
    out["elapsed_workday_ratio"] = out["anchor_wd_seq"] / out["max_wd_seq"].replace(0, np.nan)
    out["remaining_workday_ratio"] = out["remaining_workdays_after_anchor"] / out["max_wd_seq"].replace(0, np.nan)
    out["anchor_qty_per_hosp"] = out["anchor_mtd_qty"] / out["anchor_mtd_num_hosp"].replace(0, np.nan)
    out["anchor_avg_qty_per_elapsed_workday"] = out["anchor_mtd_qty"] / out["anchor_wd_seq"].replace(0, np.nan)
    out["anchor_avg_hosp_per_elapsed_workday"] = out["anchor_mtd_num_hosp"] / out["anchor_wd_seq"].replace(0, np.nan)
    return out


train_fe = add_safe_features(train_df)
valid_fe = add_safe_features(valid_df)
test_fe = add_safe_features(test_df)
all_fe = add_safe_features(all_df)

BASE_FEATURE_COLUMNS = [
    "year",
    "month_num",
    "prev_year_ym",
    "prev_2year_ym",
    "days",
    "workdays",
    "forecast_offset",
    "anchor_day",
    "anchor_dow",
    "forecast_start_day",
    "anchor_wd_seq",
    "max_wd_seq",
    "remaining_workdays_after_anchor",
    "anchor_mtd_qty",
    "anchor_mtd_num_hosp",
    "elapsed_workday_ratio",
    "remaining_workday_ratio",
    "anchor_qty_per_hosp",
    "anchor_avg_qty_per_elapsed_workday",
    "anchor_avg_hosp_per_elapsed_workday",
]

# 中文说明：硬断言防止未来结果列误入训练特征。
for col in BASE_FEATURE_COLUMNS:
    if col in LEAKAGE_COLS or col in IDENTIFIER_COLS - {"forecast_offset"} or col == CONFIG.target_col:
        raise AssertionError(f"发现泄露特征: {col}")

feature_audit = pd.DataFrame({
    "feature": BASE_FEATURE_COLUMNS,
    "train_missing_pct": [train_fe[col].isna().mean() * 100 for col in BASE_FEATURE_COLUMNS],
    "valid_missing_pct": [valid_fe[col].isna().mean() * 100 for col in BASE_FEATURE_COLUMNS],
    "test_missing_pct": [test_fe[col].isna().mean() * 100 for col in BASE_FEATURE_COLUMNS],
})
display(feature_audit)


In [ ]:
def get_xy(df: pd.DataFrame, feature_columns: list[str]) -> tuple[pd.DataFrame, pd.Series]:
    """中文说明：统一生成模型输入和标签。"""
    return df[feature_columns].copy(), df[CONFIG.target_col].copy()


def calculate_feature_ranking(train_data: pd.DataFrame, feature_columns: list[str]) -> pd.DataFrame:
    """中文说明：特征选择只在 train 上做，避免 valid/test 信息泄露。"""
    x_train, y_train = get_xy(train_data, feature_columns)
    x_train = x_train.replace([np.inf, -np.inf], np.nan)
    x_train = pd.DataFrame(SimpleImputer(strategy="median").fit_transform(x_train), columns=feature_columns)
    scores = mutual_info_regression(x_train, y_train, random_state=CONFIG.random_seed)
    ranking = pd.DataFrame({"feature": feature_columns, "mutual_info": scores})
    return ranking.sort_values("mutual_info", ascending=False).reset_index(drop=True)


feature_ranking = calculate_feature_ranking(train_fe, BASE_FEATURE_COLUMNS)
TOP_FEATURE_COLUMNS = feature_ranking.head(min(CONFIG.top_k_features, len(BASE_FEATURE_COLUMNS)))["feature"].tolist()
FEATURE_SETS = {
    "safe_all": BASE_FEATURE_COLUMNS,
    f"train_mi_top_{len(TOP_FEATURE_COLUMNS)}": TOP_FEATURE_COLUMNS,
}

display(feature_ranking)
print("特征集合:")
for name, cols in FEATURE_SETS.items():
    print(f"- {name}: {len(cols)} features -> {cols}")


In [ ]:
def month_total_mape(y_true: np.ndarray | pd.Series, y_pred: np.ndarray | pd.Series) -> float:
    """中文说明：与 eda.ipynb 完全一致的月总量 MAPE 口径。"""
    y_true_arr = np.asarray(y_true, dtype=float)
    y_pred_arr = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(y_pred_arr - y_true_arr) / y_true_arr) * 100)


def build_regression_pipeline(model: Any, scale: bool = False, log_target: bool = True) -> Pipeline | TransformedTargetRegressor:
    """中文说明：每个模型共用缺失值处理；线性/距离模型额外做标准化。"""
    steps: list[tuple[str, Any]] = [("imputer", SimpleImputer(strategy="median"))]
    if scale:
        steps.append(("scaler", StandardScaler()))
    steps.append(("model", model))
    pipeline = Pipeline(steps)
    if log_target:
        return TransformedTargetRegressor(regressor=pipeline, func=np.log1p, inverse_func=np.expm1)
    return pipeline


@dataclass(frozen=True)
class ModelSpec:
    name: str
    category: str
    estimator: Any
    param_grid: dict[str, list[Any]]


def get_fallback_model_specs() -> list[ModelSpec]:
    """中文说明：PyCaret 不可用时，使用 sklearn/xgboost 覆盖常见回归模型族。"""
    specs: list[ModelSpec] = [
        ModelSpec("linear_regression", "统计 linear", build_regression_pipeline(LinearRegression(), scale=True), {"regressor__model__fit_intercept": [True, False]}),
        ModelSpec("ridge", "统计 linear", build_regression_pipeline(Ridge(random_state=CONFIG.random_seed), scale=True), {"regressor__model__alpha": [0.1, 1.0, 10.0]}),
        ModelSpec("lasso", "统计 linear", build_regression_pipeline(Lasso(random_state=CONFIG.random_seed, max_iter=10000), scale=True), {"regressor__model__alpha": [0.001, 0.01, 0.1, 1.0]}),
        ModelSpec("elastic_net", "统计 linear", build_regression_pipeline(ElasticNet(random_state=CONFIG.random_seed, max_iter=10000), scale=True), {"regressor__model__alpha": [0.001, 0.01, 0.1], "regressor__model__l1_ratio": [0.2, 0.5, 0.8]}),
        ModelSpec("huber", "统计 linear", build_regression_pipeline(HuberRegressor(max_iter=1000), scale=True), {"regressor__model__epsilon": [1.2, 1.35, 1.5]}),
        ModelSpec("knn", "距离模型", build_regression_pipeline(KNeighborsRegressor(), scale=True), {"regressor__model__n_neighbors": [2, 3, 4], "regressor__model__weights": ["uniform", "distance"]}),
        ModelSpec("svr", "核模型", build_regression_pipeline(SVR(), scale=True), {"regressor__model__C": [0.1, 1.0, 10.0], "regressor__model__epsilon": [0.01, 0.1]}),
        ModelSpec("decision_tree", "树模型", build_regression_pipeline(DecisionTreeRegressor(random_state=CONFIG.random_seed)), {"regressor__model__max_depth": [2, 3, None], "regressor__model__min_samples_leaf": [1, 2, 3]}),
        ModelSpec("random_forest", "ML bagging", build_regression_pipeline(RandomForestRegressor(random_state=CONFIG.random_seed, n_jobs=-1)), {"regressor__model__n_estimators": [100, 300], "regressor__model__max_depth": [2, 3, None], "regressor__model__min_samples_leaf": [1, 2]}),
        ModelSpec("extra_trees", "ML bagging", build_regression_pipeline(ExtraTreesRegressor(random_state=CONFIG.random_seed, n_jobs=-1)), {"regressor__model__n_estimators": [100, 300], "regressor__model__max_depth": [2, 3, None], "regressor__model__min_samples_leaf": [1, 2]}),
        ModelSpec("bagging", "ML bagging", build_regression_pipeline(BaggingRegressor(random_state=CONFIG.random_seed, n_estimators=100)), {"regressor__model__max_samples": [0.6, 0.8, 1.0]}),
        ModelSpec("gradient_boosting", "ML boosting", build_regression_pipeline(GradientBoostingRegressor(random_state=CONFIG.random_seed)), {"regressor__model__n_estimators": [50, 100], "regressor__model__learning_rate": [0.03, 0.1], "regressor__model__max_depth": [2, 3]}),
        ModelSpec("hist_gradient_boosting", "ML boosting", build_regression_pipeline(HistGradientBoostingRegressor(random_state=CONFIG.random_seed)), {"regressor__model__learning_rate": [0.03, 0.1], "regressor__model__max_leaf_nodes": [7, 15, 31]}),
        ModelSpec("ada_boost", "ML boosting", build_regression_pipeline(AdaBoostRegressor(random_state=CONFIG.random_seed)), {"regressor__model__n_estimators": [50, 100], "regressor__model__learning_rate": [0.03, 0.1, 0.3]}),
    ]
    if XGBRegressor is not None:
        specs.append(ModelSpec(
            "xgboost",
            "ML boosting",
            build_regression_pipeline(XGBRegressor(random_state=CONFIG.random_seed, objective="reg:squarederror", n_jobs=-1)),
            {"regressor__model__n_estimators": [50, 100], "regressor__model__learning_rate": [0.03, 0.1], "regressor__model__max_depth": [2, 3]},
        ))
    return specs


fallback_model_specs = get_fallback_model_specs()
pd.DataFrame([{"model_name": s.name, "category": s.category, "n_param_candidates": len(list(ParameterGrid(s.param_grid)))} for s in fallback_model_specs])


In [ ]:
def iter_param_candidates(param_grid: dict[str, list[Any]], limit: int) -> list[dict[str, Any]]:
    """中文说明：限制每个模型的候选超参数数量，保持 notebook 运行轻量。"""
    candidates = list(ParameterGrid(param_grid)) if param_grid else [{}]
    if len(candidates) <= limit:
        return candidates
    rng = np.random.default_rng(CONFIG.random_seed)
    selected_idx = sorted(rng.choice(len(candidates), size=limit, replace=False).tolist())
    return [candidates[i] for i in selected_idx]


def time_cv_score(estimator: Any, x_train: pd.DataFrame, y_train: pd.Series, params: dict[str, Any]) -> float:
    """中文说明：在 train 内部做时间顺序 CV，不随机打乱，避免未来月份泄露到过去。"""
    cv = TimeSeriesSplit(n_splits=min(CONFIG.cv_splits, len(x_train) - 1))
    fold_scores: list[float] = []
    for fold_train_idx, fold_valid_idx in cv.split(x_train):
        fold_estimator = clone(estimator)
        fold_estimator.set_params(**params)
        x_fold_train, x_fold_valid = x_train.iloc[fold_train_idx], x_train.iloc[fold_valid_idx]
        y_fold_train, y_fold_valid = y_train.iloc[fold_train_idx], y_train.iloc[fold_valid_idx]
        fold_estimator.fit(x_fold_train, y_fold_train)
        fold_pred = fold_estimator.predict(x_fold_valid)
        fold_scores.append(month_total_mape(y_fold_valid, fold_pred))
    return float(np.mean(fold_scores))


def evaluate_model_on_split(model: Any, df: pd.DataFrame, feature_columns: list[str], split_name: str) -> pd.DataFrame:
    """中文说明：预测月总量，并强制不低于锚点已知 MTD。"""
    x, y = get_xy(df, feature_columns)
    raw_pred = model.predict(x)
    pred = np.maximum(raw_pred, df["anchor_mtd_qty"].to_numpy(dtype=float))
    pred = np.maximum(pred, 0)
    out = df[[
        "bizym", "forecast_offset", "split", "anchor_date", "forecast_start_date",
        "anchor_mtd_qty", CONFIG.target_col, "workdays", "max_wd_seq", "anchor_wd_seq",
    ]].copy()
    out["eval_split"] = split_name
    out["raw_predicted_month_total"] = raw_pred
    out["predicted_month_total"] = pred
    out["month_total_error"] = out["predicted_month_total"] - out[CONFIG.target_col]
    out["month_total_error_pct"] = out["month_total_error"] / out[CONFIG.target_col] * 100
    out["month_total_mape_pct"] = out["month_total_error_pct"].abs()
    out["month_total_accuracy_pct"] = 100 - out["month_total_mape_pct"]
    out["anchor_actual_mtd"] = out["anchor_mtd_qty"]
    out["anchor_completion_pct"] = out["anchor_actual_mtd"] / out[CONFIG.target_col] * 100
    out["predicted_remaining"] = out["predicted_month_total"] - out["anchor_actual_mtd"]
    out["actual_remaining"] = out[CONFIG.target_col] - out["anchor_actual_mtd"]
    out["remaining_error_pct"] = np.where(
        out["actual_remaining"] > 0,
        (out["predicted_remaining"] - out["actual_remaining"]) / out["actual_remaining"] * 100,
        np.nan,
    )
    return out

def summarize_eval(eval_df: pd.DataFrame) -> pd.DataFrame:
    """中文说明：按 split 汇总模型表现。"""
    return eval_df.groupby("eval_split").agg(
        n_months=("bizym", "nunique"),
        mean_mape=("month_total_mape_pct", "mean"),
        median_mape=("month_total_mape_pct", "median"),
        mean_accuracy=("month_total_accuracy_pct", "mean"),
        mean_remaining_error_pct=("remaining_error_pct", "mean"),
    ).reset_index()


def fit_one_fallback_model(spec: ModelSpec, feature_set_name: str, feature_columns: list[str]) -> dict[str, Any]:
    """中文说明：训练一个 sklearn/xgboost 模型，并返回最佳参数和三段评估结果。"""
    x_train, y_train = get_xy(train_fe, feature_columns)
    best_params: dict[str, Any] | None = None
    best_cv_mape = np.inf
    for params in iter_param_candidates(spec.param_grid, CONFIG.n_iter):
        try:
            cv_mape = time_cv_score(spec.estimator, x_train, y_train, params)
        except Exception as exc:
            continue
        if cv_mape < best_cv_mape:
            best_cv_mape = cv_mape
            best_params = params
    if best_params is None:
        raise ValueError("没有可用的超参数候选")

    best_model = clone(spec.estimator)
    best_model.set_params(**best_params)
    best_model.fit(x_train, y_train)

    eval_parts = [
        evaluate_model_on_split(best_model, train_fe, feature_columns, "train"),
        evaluate_model_on_split(best_model, valid_fe, feature_columns, "valid"),
        evaluate_model_on_split(best_model, test_fe, feature_columns, "test"),
    ]
    eval_df = pd.concat(eval_parts, ignore_index=True)
    eval_df["model_name"] = spec.name
    eval_df["model_category"] = spec.category
    eval_df["feature_set"] = feature_set_name
    eval_df["cv_mape"] = best_cv_mape
    return {
        "model_name": spec.name,
        "model_category": spec.category,
        "feature_set": feature_set_name,
        "feature_columns": feature_columns,
        "best_params": best_params,
        "cv_mape": best_cv_mape,
        "model": best_model,
        "eval_df": eval_df,
        "backend": "sklearn_fallback",
    }


def run_fallback_experiments() -> tuple[pd.DataFrame, dict[tuple[str, str], dict[str, Any]]]:
    """中文说明：运行兜底模型池，覆盖统计 linear、boosting、bagging 等常见模型。"""
    fitted_store: dict[tuple[str, str], dict[str, Any]] = {}
    rows: list[dict[str, Any]] = []
    for feature_set_name, feature_columns in FEATURE_SETS.items():
        for spec in fallback_model_specs:
            key = (spec.name, feature_set_name)
            try:
                result = fit_one_fallback_model(spec, feature_set_name, feature_columns)
                fitted_store[key] = result
                split_summary = summarize_eval(result["eval_df"])
                row = {
                    "model_name": spec.name,
                    "model_category": spec.category,
                    "feature_set": feature_set_name,
                    "backend": result["backend"],
                    "cv_mape": result["cv_mape"],
                    "best_params": result["best_params"],
                    "status": "ok",
                    "error": "",
                }
                for _, split_row in split_summary.iterrows():
                    split_name = split_row["eval_split"]
                    row[f"{split_name}_mean_mape"] = split_row["mean_mape"]
                    row[f"{split_name}_median_mape"] = split_row["median_mape"]
                    row[f"{split_name}_mean_accuracy"] = split_row["mean_accuracy"]
                rows.append(row)
            except Exception as exc:
                rows.append({
                    "model_name": spec.name,
                    "model_category": spec.category,
                    "feature_set": feature_set_name,
                    "backend": "sklearn_fallback",
                    "cv_mape": np.nan,
                    "best_params": {},
                    "status": "failed",
                    "error": str(exc),
                })
    results = pd.DataFrame(rows)
    return results.sort_values(["status", "valid_mean_mape", "cv_mape"], na_position="last"), fitted_store


In [ ]:
def run_pycaret_experiments_if_available() -> tuple[pd.DataFrame | None, dict[tuple[str, str], dict[str, Any]] | None]:
    """中文说明：如果 PyCaret 已安装，优先用 PyCaret 的全部可用回归模型做快速 benchmark。"""
    if not CONFIG.use_pycaret_if_available or importlib.util.find_spec("pycaret") is None:
        print("PyCaret 未安装或未启用，本次使用 sklearn/xgboost 兜底模型池。")
        return None, None

    from pycaret.regression import create_model, models, predict_model, pull, setup, tune_model

    pycaret_rows: list[dict[str, Any]] = []
    pycaret_store: dict[tuple[str, str], dict[str, Any]] = {}
    for feature_set_name, feature_columns in FEATURE_SETS.items():
        train_for_pycaret = train_fe[feature_columns + [CONFIG.target_col]].copy()
        setup(
            data=train_for_pycaret,
            target=CONFIG.target_col,
            session_id=CONFIG.random_seed,
            fold=CONFIG.cv_splits,
            fold_strategy=TimeSeriesSplit(n_splits=min(CONFIG.cv_splits, len(train_for_pycaret) - 1)),
            preprocess=True,
            html=False,
            verbose=False,
        )
        model_table = models()
        model_ids = model_table.index.tolist() if CONFIG.model_include == "all" else list(CONFIG.model_include)
        for model_id in model_ids:
            try:
                base_model = create_model(model_id, fold=CONFIG.cv_splits, verbose=False)
                tuned_model = tune_model(base_model, optimize="MAPE", n_iter=CONFIG.n_iter, choose_better=True, verbose=False)
                _ = pull()

                eval_parts = []
                for split_name, split_df in [("train", train_fe), ("valid", valid_fe), ("test", test_fe)]:
                    pred_input = split_df[feature_columns + [CONFIG.target_col]].copy()
                    pred_df = predict_model(tuned_model, data=pred_input, verbose=False)
                    label_col = "prediction_label" if "prediction_label" in pred_df.columns else "Label"
                    out = split_df.copy()
                    out["pycaret_prediction"] = pred_df[label_col].to_numpy()
                    evaluated = evaluate_model_on_split(
                        model=type("PyCaretWrapper", (), {"predict": lambda self, x, p=out["pycaret_prediction"].to_numpy(): p})(),
                        df=out,
                        feature_columns=feature_columns,
                        split_name=split_name,
                    )
                    eval_parts.append(evaluated)
                eval_df = pd.concat(eval_parts, ignore_index=True)
                eval_df["model_name"] = str(model_id)
                eval_df["model_category"] = "pycaret_regression"
                eval_df["feature_set"] = feature_set_name
                eval_df["cv_mape"] = np.nan
                split_summary = summarize_eval(eval_df)
                row = {
                    "model_name": str(model_id),
                    "model_category": "pycaret_regression",
                    "feature_set": feature_set_name,
                    "backend": "pycaret",
                    "cv_mape": np.nan,
                    "best_params": str(tuned_model),
                    "status": "ok",
                    "error": "",
                }
                for _, split_row in split_summary.iterrows():
                    split_name = split_row["eval_split"]
                    row[f"{split_name}_mean_mape"] = split_row["mean_mape"]
                    row[f"{split_name}_median_mape"] = split_row["median_mape"]
                    row[f"{split_name}_mean_accuracy"] = split_row["mean_accuracy"]
                pycaret_rows.append(row)
                pycaret_store[(str(model_id), feature_set_name)] = {
                    "model_name": str(model_id),
                    "model_category": "pycaret_regression",
                    "feature_set": feature_set_name,
                    "feature_columns": feature_columns,
                    "best_params": str(tuned_model),
                    "cv_mape": np.nan,
                    "model": tuned_model,
                    "eval_df": eval_df,
                    "backend": "pycaret",
                }
            except Exception as exc:
                pycaret_rows.append({
                    "model_name": str(model_id),
                    "model_category": "pycaret_regression",
                    "feature_set": feature_set_name,
                    "backend": "pycaret",
                    "cv_mape": np.nan,
                    "best_params": {},
                    "status": "failed",
                    "error": str(exc),
                })
    results = pd.DataFrame(pycaret_rows)
    return results.sort_values(["status", "valid_mean_mape"], na_position="last"), pycaret_store


pycaret_results, pycaret_store = run_pycaret_experiments_if_available()
if pycaret_results is not None and (pycaret_results["status"] == "ok").any():
    experiment_results = pycaret_results
    fitted_store = pycaret_store
else:
    experiment_results, fitted_store = run_fallback_experiments()

comparison_cols = [
    "model_name", "model_category", "feature_set", "backend", "status", "cv_mape",
    "train_mean_mape", "valid_mean_mape", "test_mean_mape", "best_params", "error",
]
display(experiment_results[[col for col in comparison_cols if col in experiment_results.columns]])


## Block 3：模型选择

本 block 只根据 valid MAPE 选择最优模型。test MAPE 只作为最终报告，不能反向影响模型或超参数选择。


In [ ]:
ok_results = experiment_results[experiment_results["status"] == "ok"].copy()
if ok_results.empty:
    raise ValueError("没有成功训练的模型，请检查依赖或特征配置")

best_row = ok_results.sort_values(["valid_mean_mape", "cv_mape"], na_position="last").iloc[0]
best_model_name = best_row["model_name"]
best_feature_set = best_row["feature_set"]
best_key = (best_model_name, best_feature_set)
best_record = fitted_store[best_key]
best_eval = best_record["eval_df"].sort_values(["bizym"]).reset_index(drop=True)

print(f"最优模型: {best_model_name}")
print(f"模型类别: {best_row['model_category']}")
print(f"特征集合: {best_feature_set}")
print(f"预测 offset: {CONFIG.forecast_offset}")
print(f"选择依据: valid_mean_mape = {best_row['valid_mean_mape']:.4f}%")
print(f"后端: {best_row['backend']}")
print("最佳参数:")
print(best_row["best_params"])

show_cols = [
    "eval_split", "forecast_offset", "bizym", "anchor_date", "anchor_wd_seq", "max_wd_seq", "anchor_completion_pct",
    "predicted_month_total", CONFIG.target_col, "month_total_error_pct", "month_total_mape_pct",
    "month_total_accuracy_pct", "remaining_error_pct",
]
display(best_eval[show_cols])


In [ ]:
split_summary = best_eval.groupby("eval_split").agg(
    n_months=("bizym", "nunique"),
    mean_month_total_mape=("month_total_mape_pct", "mean"),
    median_month_total_mape=("month_total_mape_pct", "median"),
    mean_accuracy=("month_total_accuracy_pct", "mean"),
    mean_remaining_error_pct=("remaining_error_pct", "mean"),
).reset_index()
display(split_summary)


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)
plot_df = best_eval.sort_values("bizym").copy()
plot_df["bizym_str"] = plot_df["bizym"].astype(str)

sns.lineplot(
    data=plot_df,
    x="bizym_str",
    y="month_total_mape_pct",
    hue="eval_split",
    marker="o",
    ax=axes[0],
)
axes[0].set_title(f"Best model ({best_model_name}) offset={CONFIG.forecast_offset} month-total MAPE")
axes[0].set_ylabel("MAPE %")

sns.lineplot(
    data=plot_df,
    x="bizym_str",
    y="month_total_accuracy_pct",
    hue="eval_split",
    marker="o",
    ax=axes[1],
    legend=False,
)
axes[1].set_title(f"offset={CONFIG.forecast_offset} month-total accuracy = 100 - MAPE")
axes[1].set_ylabel("Accuracy %")
axes[1].set_xlabel("bizym")

for ax in axes:
    ax.grid(alpha=0.25)
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(13, 5))
rank_plot_df = ok_results.sort_values("valid_mean_mape").head(20).copy()
rank_plot_df["model_label"] = rank_plot_df["model_name"] + " / " + rank_plot_df["feature_set"]
sns.barplot(data=rank_plot_df, x="model_label", y="valid_mean_mape", hue="model_category")
plt.title("Top models by valid selected offset month-total MAPE")
plt.xlabel("model / feature set")
plt.ylabel("Valid mean MAPE %")
plt.xticks(rotation=45, ha="right")
plt.grid(alpha=0.25, axis="y")
plt.tight_layout()
plt.show()


## Block 4：误差分析 + Insight 总结

本 block 面向最优模型，复用 `eda.ipynb` 的误差切片思路：看完成率、工作日数量、季节和高误差月份，形成下一轮实验假设。


In [ ]:
def add_error_slices(eval_df: pd.DataFrame) -> pd.DataFrame:
    """中文说明：补充误差方向和业务切片。"""
    out = eval_df.copy()
    out["abs_error_pct"] = out["month_total_mape_pct"]
    out["signed_error_direction"] = np.where(out["month_total_error_pct"] >= 0, "over_forecast", "under_forecast")
    out["completion_bucket"] = pd.cut(
        out["anchor_completion_pct"],
        bins=[0, 60, 70, 80, 90, 100, np.inf],
        labels=["<=60%", "60-70%", "70-80%", "80-90%", "90-100%", ">100%"],
        include_lowest=True,
    )
    out["workday_count_bucket"] = pd.cut(
        out["workdays"],
        bins=[0, 19, 20, 21, 22, 40],
        labels=["<=19", "20", "21", "22", ">=23"],
        include_lowest=True,
    )
    out["month_num"] = out["bizym"] % 100
    out["season"] = np.select(
        [
            out["month_num"].isin([12, 1, 2]),
            out["month_num"].isin([3, 4, 5]),
            out["month_num"].isin([6, 7, 8]),
            out["month_num"].isin([9, 10, 11]),
        ],
        ["winter", "spring", "summer", "autumn"],
        default="unknown",
    )
    return out


def summarize_slice(df_slice: pd.DataFrame, col: str) -> pd.DataFrame:
    """中文说明：按切片汇总 MAPE 和高估比例。"""
    return df_slice.groupby(col, dropna=False).agg(
        n_months=("bizym", "nunique"),
        mean_mape=("abs_error_pct", "mean"),
        median_mape=("abs_error_pct", "median"),
        over_forecast_rate=("signed_error_direction", lambda s: (s == "over_forecast").mean() * 100),
    ).reset_index().sort_values("mean_mape", ascending=False)


error_df = add_error_slices(best_eval)
top_error = error_df.sort_values("abs_error_pct", ascending=False).head(12)
display(top_error[["eval_split", "forecast_offset", "bizym", "abs_error_pct", "signed_error_direction", "completion_bucket", "workday_count_bucket", "season", "predicted_month_total", CONFIG.target_col]])


In [ ]:
for slice_col in ["eval_split", "completion_bucket", "workday_count_bucket", "season", "signed_error_direction"]:
    print(f"\n=== Error slice: {slice_col} ===")
    display(summarize_slice(error_df, slice_col))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.scatterplot(
    data=error_df,
    x="anchor_completion_pct",
    y="abs_error_pct",
    hue="eval_split",
    style="signed_error_direction",
    s=90,
    ax=axes[0],
)
axes[0].set_title("Error vs anchor completion %")
axes[0].set_xlabel("Anchor completion %")
axes[0].set_ylabel("Month-total MAPE %")

sns.scatterplot(
    data=error_df,
    x=CONFIG.target_col,
    y="abs_error_pct",
    hue="eval_split",
    style="signed_error_direction",
    s=90,
    ax=axes[1],
)
axes[1].set_title("Error vs actual monthly total")
axes[1].set_xlabel("Actual month total")
axes[1].set_ylabel("Month-total MAPE %")
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x/1e6:.1f}M"))
for ax in axes:
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
def build_insight_summary(error_data: pd.DataFrame) -> str:
    """中文说明：生成可复制到实验记录里的中文 insight 摘要。"""
    split_perf = error_data.groupby("eval_split")["abs_error_pct"].mean().sort_values()
    worst_month = error_data.sort_values("abs_error_pct", ascending=False).iloc[0]
    direction_rate = error_data["signed_error_direction"].value_counts(normalize=True).mul(100)
    worst_bucket = summarize_slice(error_data, "completion_bucket").iloc[0]
    lines = [
        "### Insight 总结",
        f"- 最优模型为 `{best_model_name}`，使用 `{best_feature_set}` 特征集合；选择依据是 valid 平均 MAPE 最低。",
        f"- split 平均 MAPE：" + ", ".join([f"{idx}={val:.2f}%" for idx, val in split_perf.items()]) + "。",
        f"- 最大误差月份是 {int(worst_month['bizym'])}，split={worst_month['eval_split']}，MAPE={worst_month['abs_error_pct']:.2f}%，方向={worst_month['signed_error_direction']}。",
        f"- 误差最高的完成率切片是 {worst_bucket['completion_bucket']}，平均 MAPE={worst_bucket['mean_mape']:.2f}%，样本数={int(worst_bucket['n_months'])}。",
        f"- 高估/低估比例：" + ", ".join([f"{idx}={val:.1f}%" for idx, val in direction_rate.items()]) + "。",
        "- 下一轮建议：补充更多历史月份后重新评估模型稳定性；优先验证剩余量 target、历史同月 lag 特征、以及节假日结构特征是否能改善高误差切片。",
    ]
    return "\n".join(lines)

insight_markdown = build_insight_summary(error_df)
print(insight_markdown)


In [ ]:
# 高 MAPE 月份预测路径复盘：调整该阈值即可控制展示范围，单位是百分比。
MAPE_THRESHOLD_PCT = 10.0


def resolve_daily_sales_path() -> Path:
    """中文说明：兼容从仓库根目录或 notebook 目录运行。"""
    candidates = [base / 'data' / 'sales_30d_daily.csv' for base in (Path.cwd(), *Path.cwd().parents)]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"找不到日级销量数据，已尝试: {[str(p) for p in candidates]}")


def load_daily_sales_for_path_plot(path: Path) -> pd.DataFrame:
    """中文说明：读取日级销量，并按 01-reproduction.ipynb 的口径补齐日历和重算 MTD。"""
    if importlib.util.find_spec("chinese_calendar") is None:
        raise ImportError("缺少依赖 chinese_calendar。请先安装：pip install chinese-calendar")
    from chinese_calendar import is_workday as cn_is_workday

    raw = pd.read_csv(path)
    required = {"bizym", "transdate", "num_hosp", "qty"}
    missing = sorted(required - set(raw.columns))
    if missing:
        raise ValueError(f"{path} 缺少必要字段: {missing}")

    raw = raw[["bizym", "transdate", "num_hosp", "qty"]].copy()
    raw["transdate"] = pd.to_datetime(raw["transdate"], errors="coerce")
    raw["bizym"] = raw["bizym"].astype(int)
    raw["qty"] = pd.to_numeric(raw["qty"], errors="coerce")
    raw["num_hosp"] = pd.to_numeric(raw["num_hosp"], errors="coerce")
    if raw[["transdate", "qty", "num_hosp"]].isna().any().any():
        raise ValueError(f"{path} 存在无法解析的日期或数值")

    parts = []
    for ym, group in raw.groupby("bizym", sort=True):
        year, month = divmod(int(ym), 100)
        month_dates = pd.date_range(f"{year:04d}-{month:02d}-01", periods=pd.Period(f"{year:04d}-{month:02d}").days_in_month, freq="D")
        cal = pd.DataFrame({"transdate": month_dates})
        cal["day_of_month"] = cal["transdate"].dt.day
        cal["is_workday"] = cal["transdate"].dt.date.map(lambda d: bool(cn_is_workday(d)))
        cal["wd_seq"] = cal["is_workday"].cumsum().where(cal["is_workday"], 0).astype(int)

        merged = cal.merge(group[["transdate", "qty", "num_hosp"]], on="transdate", how="left")
        merged[["qty", "num_hosp"]] = merged[["qty", "num_hosp"]].fillna(0.0)
        merged["bizym"] = int(ym)

        workday_indices = merged.index[merged["is_workday"]].tolist()
        for idx in merged.index[~merged["is_workday"]]:
            qty_val = float(merged.at[idx, "qty"])
            hosp_val = float(merged.at[idx, "num_hosp"])
            if qty_val == 0 and hosp_val == 0:
                continue
            next_wd = [i for i in workday_indices if i > idx]
            prev_wd = [i for i in workday_indices if i < idx]
            if not next_wd and not prev_wd:
                continue
            target_idx = next_wd[0] if next_wd else prev_wd[-1]
            merged.at[target_idx, "qty"] += qty_val
            merged.at[target_idx, "num_hosp"] += hosp_val
            merged.at[idx, "qty"] = 0.0
            merged.at[idx, "num_hosp"] = 0.0

        merged["mtd_qty"] = merged["qty"].cumsum()
        merged["mtd_num_hosp"] = merged["num_hosp"].cumsum()
        parts.append(merged[["bizym", "transdate", "day_of_month", "is_workday", "wd_seq", "qty", "num_hosp", "mtd_qty", "mtd_num_hosp"]])

    return pd.concat(parts, ignore_index=True).sort_values(["bizym", "transdate"]).reset_index(drop=True)


def build_month_total_forecast_path(data: pd.DataFrame, row: pd.Series) -> pd.DataFrame:
    """中文说明：把月总量预测还原为日级 MTD 路径；工作日固定斜率，非工作日延续上一点。"""
    target_ym = int(row["bizym"])
    month_df = data[data["bizym"] == target_ym].sort_values("transdate").copy()
    if month_df.empty:
        raise ValueError(f"日级数据中找不到 bizym={target_ym}")

    anchor_date = pd.Timestamp(row["anchor_date"])
    predicted_month_total = float(row["predicted_month_total"])
    anchor_mtd = float(row.get("anchor_actual_mtd", row.get("anchor_mtd_qty", np.nan)))
    if not np.isfinite(anchor_mtd):
        anchor_match = month_df.loc[month_df["transdate"] <= anchor_date, "mtd_qty"]
        if anchor_match.empty:
            raise ValueError(f"bizym={target_ym} 找不到 anchor_date={anchor_date.date()} 之前的 MTD")
        anchor_mtd = float(anchor_match.iloc[-1])

    out = month_df[["transdate", "day_of_month", "is_workday", "wd_seq"]].copy()
    out["predicted_mtd_qty"] = np.nan
    out.loc[out["transdate"] == anchor_date, "predicted_mtd_qty"] = anchor_mtd

    after_anchor = out["transdate"] > anchor_date
    future_workday = after_anchor & out["is_workday"]
    remaining_workdays = int(future_workday.sum())
    remaining_qty = predicted_month_total - anchor_mtd

    if remaining_workdays > 0:
        future_rank = out.loc[future_workday, "is_workday"].cumsum().to_numpy(dtype=float)
        out.loc[future_workday, "predicted_mtd_qty"] = anchor_mtd + remaining_qty * future_rank / remaining_workdays
    else:
        out.loc[after_anchor, "predicted_mtd_qty"] = predicted_month_total

    out["predicted_mtd_qty"] = out["predicted_mtd_qty"].ffill()
    out.loc[out["transdate"] < anchor_date, "predicted_mtd_qty"] = np.nan
    out["predicted_mtd_qty"] = out["predicted_mtd_qty"].round()
    out.loc[out.index[-1], "predicted_mtd_qty"] = predicted_month_total
    return out[["transdate", "predicted_mtd_qty"]]


def plot_high_mape_forecast_path(data: pd.DataFrame, forecast_df: pd.DataFrame, target_ym: int, title: str) -> None:
    """中文说明：保持 01-reproduction.ipynb 的图形风格。"""
    actual = data[data["bizym"] == int(target_ym)][["transdate", "day_of_month", "mtd_qty"]].copy()
    merged = actual.merge(forecast_df[["transdate", "predicted_mtd_qty"]], on="transdate", how="left")
    fig, ax = plt.subplots(figsize=(12, 4.5))
    ax.plot(merged["day_of_month"], merged["mtd_qty"], marker="o", linewidth=2, label="Actual MTD")
    ax.plot(merged["day_of_month"], merged["predicted_mtd_qty"], marker="s", linestyle="--", linewidth=2, label="Forecast MTD")
    ax.set_title(title)
    ax.set_xlabel("Day of month")
    ax.set_ylabel("MTD qty")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    ax.legend()
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()


daily_df_for_path = load_daily_sales_for_path_plot(resolve_daily_sales_path())
high_mape_eval = best_eval[best_eval["month_total_mape_pct"] > MAPE_THRESHOLD_PCT].copy()
split_order = {"train": 0, "valid": 1, "test": 2}
high_mape_eval["_split_order"] = high_mape_eval["eval_split"].map(split_order).fillna(99)
high_mape_eval = high_mape_eval.sort_values(["_split_order", "bizym"]).drop(columns="_split_order").reset_index(drop=True)

print(f"MAPE_THRESHOLD_PCT = {MAPE_THRESHOLD_PCT:.2f}%")
print(f"命中月份数: {len(high_mape_eval)} / {len(best_eval)}")
display(high_mape_eval[["eval_split", "forecast_offset", "bizym", "month_total_mape_pct", "predicted_month_total", CONFIG.target_col]])

for _, row in high_mape_eval.iterrows():
    ym = int(row["bizym"])
    mape = float(row["month_total_mape_pct"])
    split_name = row["eval_split"]
    forecast_path = build_month_total_forecast_path(daily_df_for_path, row)
    plot_high_mape_forecast_path(
        daily_df_for_path,
        forecast_path,
        ym,
        f"{ym} split={split_name} offset={int(row['forecast_offset'])} MAPE={mape:.2f}% forecast path - {best_model_name}",
    )


In [ ]:
high_error_threshold = error_df["abs_error_pct"].quantile(0.75)
low_error_threshold = error_df["abs_error_pct"].quantile(0.25)
high_error = error_df[error_df["abs_error_pct"] >= high_error_threshold].copy()
low_error = error_df[error_df["abs_error_pct"] <= low_error_threshold].copy()

direction_rate = error_df["signed_error_direction"].value_counts(normalize=True).mul(100)
mean_signed_error = error_df["month_total_error_pct"].mean()
worst_signed_bias = error_df.iloc[error_df["month_total_error_pct"].abs().argmax()]
actual_month_col = CONFIG.target_col

direction_label = {
    "over_forecast": "高估",
    "under_forecast": "低估",
}

insights = []
insights.append(f"高误差阈值（前 25%）= {high_error_threshold:.2f}% MAPE；低误差阈值（后 25%）= {low_error_threshold:.2f}% MAPE。")
insights.append(f"高误差月份主要是：{', '.join(map(str, high_error.sort_values('abs_error_pct', ascending=False)['bizym'].head(8).tolist()))}。")
insights.append(f"低误差月份主要是：{', '.join(map(str, low_error.sort_values('abs_error_pct')['bizym'].head(8).tolist()))}。")

for col, label in [
    ("eval_split", "评估集"),
    ("completion_bucket", "锚点完成率分桶"),
    ("workday_count_bucket", "工作日数量"),
    ("season", "季节"),
    ("signed_error_direction", "预测方向"),
]:
    summary = summarize_slice(error_df, col)
    top = summary.iloc[0]
    top_value = direction_label.get(top[col], top[col])
    insights.append(f"按{label}看，平均误差最大的分组是 {top_value}，平均 MAPE={top['mean_mape']:.2f}%，覆盖 {int(top['n_months'])} 个月。")

insights.append(
    f"整体预测偏差为平均 signed error={mean_signed_error:.2f}%；"
    f"高估占比={direction_rate.get('over_forecast', 0):.1f}%，"
    f"低估占比={direction_rate.get('under_forecast', 0):.1f}%。"
)
insights.append(
    f"绝对 signed bias 最大的月份是 {int(worst_signed_bias['bizym'])}："
    f"signed error={worst_signed_bias['month_total_error_pct']:.2f}%，"
    f"MAPE={worst_signed_bias['abs_error_pct']:.2f}%，"
    f"预测值={worst_signed_bias['predicted_month_total']:,.0f}，"
    f"实际值={worst_signed_bias[actual_month_col]:,.0f}。"
)

print("误差 insights:")
for i, item in enumerate(insights, start=1):
    print(f"{i}. {item}")